In [ ]:
!pip install transformers[torch] sentence-transformers datasets
!pip install -U accelerate
!pip install -U transformers
!pip install -U evaluate
!pip install -U peft
!pip install seqeval
!pip install torchmetrics
!pip install torcheval
!pip install pyconll

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
import torch
import transformers
import nltk
from nltk.corpus import wordnet,subjectivity,stopwords
from nltk.tag.perceptron import PerceptronTagger
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
nltk.download("subjectivity")
nltk.download('stopwords')

[nltk_data] Downloading package subjectivity to /root/nltk_data...
[nltk_data]   Unzipping corpora/subjectivity.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
import os
file_path_dev = "en_gum-ud-dev.conllu"
file_path_train = "en_gum-ud-train.conllu"
file_path_test = "en_gum-ud-test.conllu"
print(os.getcwd())
import pyconll
def read_conllu(path):
    data = pyconll.load_from_file(path)
    tagged_sentences=[]
    t=0
    for sentence in data:
        tagged_sentence=[]
        for token in sentence:
            if token.upos:
                t+=1
                tagged_sentence.append((token.form.lower(), token.upos))
        tagged_sentences.append(tagged_sentence)
    return tagged_sentences
data_dev = read_conllu(file_path_dev)
data_train = read_conllu(file_path_train)
data_test = read_conllu(file_path_test)

print(len(data_dev))
print(len(data_train))
print(len(data_test))
print(data_dev[0],data_train[0],data_test[0])
def load_ud_treebank_data(sentences):
    # Sample format: (word, pos_tag)

    words = []
    pos_tags = []

    for sentence in sentences:
        for word, pos in sentence:
            words.append(word)
            pos_tags.append(pos)

    return words, pos_tags
X_dev,Y_dev = load_ud_treebank_data(data_dev)
X_train,Y_train = load_ud_treebank_data(data_train)
X_test,Y_test = load_ud_treebank_data(data_test)

print(X_dev[0],Y_dev[0])
print(X_train[0],Y_train[0])
print(X_test[0],Y_test[0])

str_labels = set(Y_train + Y_dev + Y_test)

/content
1341
9520
1285
[('introduction', 'NOUN')] [('aesthetic', 'ADJ'), ('appreciation', 'NOUN'), ('and', 'CCONJ'), ('spanish', 'ADJ'), ('art', 'NOUN'), (':', 'PUNCT')] [('the', 'DET'), ('prevalence', 'NOUN'), ('of', 'ADP'), ('discrimination', 'NOUN'), ('across', 'ADP'), ('racial', 'ADJ'), ('groups', 'NOUN'), ('in', 'ADP'), ('contemporary', 'ADJ'), ('america', 'PROPN'), (':', 'PUNCT')]
introduction NOUN
aesthetic ADJ
the DET


In [ ]:
print(str_labels)

{'PRON', 'PART', 'AUX', 'PUNCT', 'DET', 'INTJ', 'PROPN', 'NUM', 'CCONJ', 'X', 'ADV', 'ADP', 'VERB', 'SCONJ', 'NOUN', 'ADJ', 'SYM'}


In [ ]:
id2label = {}
label2id = {}
for l,ind in enumerate(str_labels):
    id2label[l] = ind
    label2id[ind] = l

print(id2label)
print(label2id)

{0: 'PRON', 1: 'PART', 2: 'AUX', 3: 'PUNCT', 4: 'DET', 5: 'INTJ', 6: 'PROPN', 7: 'NUM', 8: 'CCONJ', 9: 'X', 10: 'ADV', 11: 'ADP', 12: 'VERB', 13: 'SCONJ', 14: 'NOUN', 15: 'ADJ', 16: 'SYM'}
{'PRON': 0, 'PART': 1, 'AUX': 2, 'PUNCT': 3, 'DET': 4, 'INTJ': 5, 'PROPN': 6, 'NUM': 7, 'CCONJ': 8, 'X': 9, 'ADV': 10, 'ADP': 11, 'VERB': 12, 'SCONJ': 13, 'NOUN': 14, 'ADJ': 15, 'SYM': 16}


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import DatasetDict,load_dataset,ClassLabel,Dataset
MODEL = "prajjwal1/bert-small"
DATASET = "batterydata/pos_tagging"
dataset = load_dataset("batterydata/pos_tagging")
# num_classes = len(id2label.keys())
labels = set()
for data in dataset['train']:
    new_labels = set(data["labels"])
    labels.update(new_labels)
for data in dataset['test']:
    new_labels = set(data["labels"])
    labels.update(new_labels)
num_classes = len(labels)
id2label = {}
label2id = {}
for ind,l in enumerate(labels):
    id2label[ind] = l
    label2id[l] = ind


test_dataset = dataset['test']

# Perform stratified split using the label
split_data = test_dataset.train_test_split(
    test_size=0.5,
)

# Extract the new splits
validation_dataset = split_data['test']
test_dataset = split_data['train']

dataset = DatasetDict({
    'train': dataset['train'],
    'validation': validation_dataset,
    'test': test_dataset,
})

def convert_pos_tags_to_ids(example):
    example["labels"] = [label2id[label] for label in example["labels"]]
    return example
dataset = dataset.map(convert_pos_tags_to_ids)
# dataset = dataset.cast_column("labels", ClassLabel(names=list(label2id.keys())))
# for data in dataset["train"]:
#     data["labels"] = [label2id[label] for label in data["labels"]]


# for data in dataset["test"]:
#     data["labels"] = [label2id[label] for label in data["labels"]]

# for data in dataset["validation"]:
#     data["labels"] = [label2id[label] for label in data["labels"]]


tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=num_classes, id2label = id2label, label2id = label2id)

print(dataset,len(labels),labels)
print(dataset["train"][0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

train.json:   0%|          | 0.00/5.05M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/601k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13054 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/116M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at prajjwal1/bert-small and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DatasetDict({
    train: Dataset({
        features: ['words', 'labels'],
        num_rows: 13054
    })
    validation: Dataset({
        features: ['words', 'labels'],
        num_rows: 726
    })
    test: Dataset({
        features: ['words', 'labels'],
        num_rows: 725
    })
}) 48 {':', 'VBN', 'CD', 'UH', 'RP', 'NN', 'RBR', '``', 'WDT', 'WP', 'JJR', '-RRB-', 'NNPS', 'PDT', 'TO', 'JJ', 'NNP', 'PRP', '#', 'WP$', 'VBZ', 'VBP', 'FW', 'MD', 'VB', 'CC', 'IN', 'JJS', 'PRP$', 'RB', 'DT', 'WRB', 'POS', 'RBS', "''", '(', ',', '$', 'SYM', 'VBD', '-LRB-', 'VBG', ')', '.', 'NNS', 'LS', '-NONE-', 'EX'}
{'words': ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], 'labels': [5, 26, 30, 5, 20, 29, 1, 14, 24, 30, 15, 5,

In [ ]:
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, 

In [ ]:
# Freeze the base model (DistilRoBERTa)
for param in model.base_model.parameters():
    param.requires_grad = False

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

Total parameters: 28525616
Trainable parameters: 24624


In [ ]:
# def preprocess_function(examples):
#     return tokenizer(examples["words"], truncation=True,is_split_into_words=True,padding="max_length",max_length=512)
# tokenized_ud = dataset.map(preprocess_function, batched=True)

def tokenize_and_align_labels(example, max_length=128):
    # Tokenize the words (using the tokenizer)
    encoding = tokenizer(example['words'], truncation=True, padding=True,max_length=max_length, is_split_into_words=True)

    # Align the labels with tokenized words
    labels = example['labels']
    # Ensure that the length of the labels matches the tokenized length
    if len(labels) < len(encoding['input_ids']):
        # Add a placeholder label (e.g., -100) for padding tokens
        labels = labels + (['0'] * (len(encoding['input_ids']) - len(labels)))  # -100 is the ignored index for labels in Hugging Face
    else:
        # If labels exceed tokenized length, truncate the labels
        labels = labels[:len(encoding['input_ids'])]

    encoding['labels'] = labels
    return encoding

# Apply tokenization and truncation
train_data = dataset["train"].map(tokenize_and_align_labels,batched=True)
validation_data = dataset["validation"].map(tokenize_and_align_labels,batched=True)
test_data = dataset["test"].map(tokenize_and_align_labels,batched=True)

# dataset = Dataset.from_list(processed_data)
dataset_new = DatasetDict({
    "train":train_data,
    "validation":validation_data,
    "test":test_data
})

# dataset_new = dataset_new.cast_column("labels",ClassLabel(names=list(label2id.values())))
print(dataset_new)

Map:   0%|          | 0/13054 [00:00<?, ? examples/s]

Map:   0%|          | 0/726 [00:00<?, ? examples/s]

Map:   0%|          | 0/725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['words', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 13054
    })
    validation: Dataset({
        features: ['words', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 726
    })
    test: Dataset({
        features: ['words', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 725
    })
})


In [ ]:
print(dataset_new)
print(dataset_new["train"][0])

DatasetDict({
    train: Dataset({
        features: ['words', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 13054
    })
    validation: Dataset({
        features: ['words', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 726
    })
    test: Dataset({
        features: ['words', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 725
    })
})
{'words': ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], 'labels': [5, 26, 30, 5, 20, 29, 1, 14, 24, 30, 15, 5, 26, 5, 44, 26, 16, 36, 15, 26, 5, 5, 36, 24, 14, 24, 30, 15, 5, 26, 16, 25, 16, 32, 15, 44, 43], 'input_ids': [101, 7023, 1999, 1996, 9044, 2003, 4235, 3517, 2000, 2202,

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

import torch

def compute_metrics(p):
    predictions, labels = p

    # Ensure predictions are converted to a PyTorch tensor
    if not isinstance(predictions, torch.Tensor):
        predictions = torch.tensor(predictions)

    # Convert logits to predicted class indices
    predictions = torch.argmax(predictions, dim=1)
    # Compute metrics
    accuracy = accuracy_metric.compute(predictions=predictions.numpy(), references=labels)
    f1 = f1_metric.compute(predictions=predictions.numpy(), references=labels, average="macro")

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt") # Dynamically pads the sequences within each patch to avoid any shape misalignments

training_args = TrainingArguments(
    output_dir='./txt_cls_frozen_base/',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

)

unique_val_labels = set()
for example in dataset_new["test"]:
    unique_val_labels.update(example["labels"])
print(len(unique_val_labels))

trainer = Trainer(
    model,
    training_args,
    train_dataset=dataset_new["train"],
    eval_dataset=dataset_new["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


43


<ipython-input-12-f8417b257495>:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`labels` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [ ]:
predictions = trainer.predict(test_dataset=dataset["test"])

# Unpack the results
logits = predictions.predictions
labels = predictions.label_ids
metrics = predictions.metrics

metrics